# EDA BigVul — шаг 0, день 1

**Вход:** `data/raw/bigvul.csv` — поля: `cve, cwe, project, commit_id, lang, func_before, split`

**Выход:** отчёт с цифрами внутри ноутбука + `data/processed/dedup_input.csv` (вход для шага дедупа)

Пять вопросов: объём данных; валидность CWE (уровень 0); распределение CWE; дубликаты и утечка train→test; проекты.

## 0. Загрузка

In [1]:
import os
import re
import pandas as pd


os.makedirs("reports", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

path = "../../lora_research_to_aij/raw_data/bigvul_func_before.jsonl"

df_raw = pd.read_json(path, lines=True)

print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.head()

(10895, 7)
['cve', 'cwe', 'project', 'commit_id', 'lang', 'func_before', 'split']


,cve,cwe,project,commit_id,lang,func_before,split
0,CVE-2017-11462,CWE-415,krb5,56f7b1bc95a2a3eeb420e069e7655fb181ade5cf,C,"gss_get_mic (minor_status,\n\t context_han...",train
1,CVE-2018-6063,CWE-787,Chrome,673ce95d481ea9368c4d4d43ac756ba1d6d9e608,C,void VideoCaptureImpl::OnBufferCreated(int32_t...,train
2,CVE-2017-13031,CWE-125,tcpdump,2d669862df7cd17f539129049f6fb70d17174125,C,"frag6_print(netdissect_options *ndo, register ...",train
3,CVE-2013-0917,CWE-119,Chrome,02c8303512ebed345011f7b545e2f418799be2f0,C,EntrySync* EntrySync::copyTo(DirectoryEntrySyn...,train
4,CVE-2015-1215,CWE-119,Chrome,2bceda4948deeaed0a5a99305d0d488eb952f64f,C,ScriptPromise BluetoothRemoteGATTServer::getPr...,train


## 1. Объём и полнота

In [2]:
print("пропуски по колонкам:")
print(df_raw.isna().sum())

print("\nраспределение по split:")
print(df_raw["split"].value_counts())

print("\nязыки (топ-10):")
print(df_raw["lang"].value_counts().head(10))

# план — C-код, остальное убираем
df_raw = df_raw[df_raw["lang"] == "C"].copy()
print(f"\nпосле фильтра lang == 'C': {len(df_raw)}")

пропуски по колонкам:
cve            0
cwe            0
project        0
commit_id      0
lang           0
func_before    0
split          0
dtype: int64

распределение по split:
split
train         8714
validation    1155
test          1026
Name: count, dtype: int64

языки (топ-10):
lang
C      10777
CPP      118
Name: count, dtype: int64

после фильтра lang == 'C': 10777


## 2. Валидность CWE (уровень 0)

Правила:
1. Метка парсится как `CWE-<число>` — отсекаем `NVD-CWE-Other`, `NVD-CWE-noinfo`, пустые.
2. CWE входит в нашу BFCWE-таблицу — иначе цепочку не построить, пример бесполезен для таргета.
3. Класс имеет ≥ 15 примеров — иначе нечему учиться.

In [3]:
def parse_cwe(x):
    m = re.match(r"^CWE-(\d+)$", str(x).strip())
    return int(m.group(1)) if m else None

df_raw["cwe_id"] = df_raw["cwe"].map(parse_cwe)  # None станет NaN, колонка станет float — это норма
invalid = df_raw["cwe_id"].isna()
print(f"невалидный формат CWE: {invalid.sum()} ({invalid.mean():.1%})")
print(df_raw.loc[invalid, "cwe"].value_counts().head())

невалидный формат CWE: 2124 (19.7%)
cwe
    2124
Name: count, dtype: int64


In [ ]:
# Проверка против BFCWE: положите таблицу в data/raw/bfcwe.csv (колонка cwe с числовыми id)
try:
    bf_cwes = set(int(x) for x in pd.read_csv("../raw_data/bfcwe.csv")["cwe"])
    df_raw["in_bfcwe"] = df_raw["cwe_id"].isin(bf_cwes)
    n_out = int((~df_raw["in_bfcwe"] & df_raw["cwe_id"].notna()).sum())
    print(f"CWE вне BFCWE-таблицы: {n_out}")
except FileNotFoundError:
    df_raw["in_bfcwe"] = True
    print("ВНИМАНИЕ: data/raw/bfcwe.csv не найдена — проверка №2 пропущена, добавьте таблицу и перезапустите")

NameError: name 'df' is not defined

## 3. Распределение CWE

In [5]:
vc = df_raw.loc[df_raw["cwe_id"].notna(), "cwe_id"].value_counts()

print(f"уникальных CWE: {len(vc)}")
print(f"топ-10 CWE покрывают {vc.head(10).sum() / vc.sum():.1%} примеров\n")
print(vc.head(10).to_string())

n_small = int((vc < 15).sum())
print(f"\nклассов с <15 примерами: {n_small} (примеров в них: {int(vc[vc < 15].sum())})")
print("эти классы — кандидаты на отсечение на шаге дедупа")

уникальных CWE: 84
топ-10 CWE покрывают 78.7% примеров

cwe_id
119.0    2132
20.0     1100
399.0     734
125.0     625
200.0     506
264.0     492
416.0     321
189.0     321
190.0     310
362.0     266

классов с <15 примерами: 49 (примеров в них: 235)
эти классы — кандидаты на отсечение на шаге дедупа


## 4. Дубликаты и утечка train → test

Нормализация лёгкая: комментарии и пробелы. Канонизация имён переменных и MinHash — завтра, по результатам этого EDA.

In [6]:
def normalize(code: str) -> str:
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.S)  # блок-комментарии
    code = re.sub(r"//[^\n]*", " ", code)                 # строчные комментарии
    return re.sub(r"\s+", " ", code).strip()              # пробелы

df_raw["norm"] = df_raw["func_before"].map(normalize)
df_raw["hash"] = df_raw["norm"].map(hash)

total_dup = int(df_raw.duplicated("hash").sum())
print(f"точных дубликатов после нормализации: {total_dup} ({total_dup / len(df_raw):.1%})")

точных дубликатов после нормализации: 1729 (16.0%)


In [7]:
# утечка: одна и та же функция в train и test -> тестовые метрики завышены
tr = set(df_raw.loc[df_raw["split"] == "train", "hash"])
te = set(df_raw.loc[df_raw["split"] == "test", "hash"])
leak = tr & te
print(f"пересечение train/test по хэшу: {len(leak)} функций")
print("-> при ненулевом значении кластеры дубликатов целиком уходят в один сплит (делается на шаге дедупа)")

пересечение train/test по хэшу: 822 функций
-> при ненулевом значении кластеры дубликатов целиком уходят в один сплит (делается на шаге дедупа)


In [8]:
# пример кластера дубликатов — глазами убедиться, что это правда дубликаты
g = df_raw.groupby("hash").size().sort_values(ascending=False)
print("размеры кластеров (топ-5):", g.head(5).tolist())

top_hash = g.index[0]
cols = [c for c in ["cve", "cwe", "project", "commit_id", "split"] if c in df_raw.columns]
df_raw[df_raw["hash"] == top_hash][cols]

размеры кластеров (топ-5): [3, 3, 2, 2, 2]


,cve,cwe,project,commit_id,split
5377,CVE-2016-3756,CWE-20,Android,659030a2e80c38fb8da0a4eb68695349eec6778b,train
6043,CVE-2017-0814,CWE-200,Android,eeb4e45d5683f88488c083ecf142dc89bc3f0b47,train
10495,CVE-2016-3756,CWE-20,Android,659030a2e80c38fb8da0a4eb68695349eec6778b,test


## 5. Проекты

In [9]:
print(f"проектов: {df_raw['project'].nunique()}")
print(df_raw["project"].value_counts().head(10).to_string())
print("\nкандидаты на проблему дисбаланса: один проект не должен давать >30% данных")

проектов: 271
project
Chrome         3951
linux          1932
Android        1283
ImageMagick     366
php-src         231
tcpdump         211
openssl         155
krb5            138
php             136
qemu            122

кандидаты на проблему дисбаланса: один проект не должен давать >30% данных


## 6. Итог и сохранение

Чек-лист перед шагом дедупа:
- [ ] доля невалидных CWE известна
- [ ] BFCWE-таблица на месте, проверка №2 пройдена
- [ ] утечка train/test измерена (если > 0 — чиним на дедупе)
- [ ] доля дубликатов сопоставима с ожиданием 30–50%
- [ ] примеры кластеров посмотрены глазами

In [10]:
out = df_raw[df_raw["cwe_id"].notna() & df_raw["in_bfcwe"]].copy()
out.to_csv("data/processed/dedup_input.csv", index=False)

summary = {
    "строк всего": len(pd.read_json(path, lines=True)),
    "после фильтра C": len(df_raw),
    "невалидный CWE": int(invalid.sum()),
    "вне BFCWE": int((~df_raw["in_bfcwe"] & df_raw["cwe_id"].notna()).sum()),
    "точных дубликатов": total_dup,
    "утечка train/test": len(leak),
    "на выходе (dedup_input.csv)": len(out),
}
for k, v in summary.items():
    print(f"{k}: {v}")
pd.Series(summary).to_csv("reports/eda_summary.csv", header=False)

строк всего: 10895
после фильтра C: 10777
невалидный CWE: 2124
вне BFCWE: 4771
точных дубликатов: 1729
утечка train/test: 822
на выходе (dedup_input.csv): 3882


In [15]:
pd.read_csv("../eda/data/processed/dedup_input.csv")

,cve,cwe,project,commit_id,lang,func_before,split,cwe_id,in_bfcwe,norm,hash
0,CVE-2017-11462,CWE-415,krb5,56f7b1bc95a2a3eeb420e069e7655fb181ade5cf,C,"gss_get_mic (minor_status,\n\t context_han...",train,415.0,True,"gss_get_mic (minor_status, context_handle, qop...",-3026410707913907481
1,CVE-2018-6063,CWE-787,Chrome,673ce95d481ea9368c4d4d43ac756ba1d6d9e608,C,void VideoCaptureImpl::OnBufferCreated(int32_t...,train,787.0,True,void VideoCaptureImpl::OnBufferCreated(int32_t...,-1992117681761256476
2,CVE-2017-13031,CWE-125,tcpdump,2d669862df7cd17f539129049f6fb70d17174125,C,"frag6_print(netdissect_options *ndo, register ...",train,125.0,True,"frag6_print(netdissect_options *ndo, register ...",7769392586341661073
3,CVE-2013-0917,CWE-119,Chrome,02c8303512ebed345011f7b545e2f418799be2f0,C,EntrySync* EntrySync::copyTo(DirectoryEntrySyn...,train,119.0,True,EntrySync* EntrySync::copyTo(DirectoryEntrySyn...,-5151006162919973459
4,CVE-2015-1215,CWE-119,Chrome,2bceda4948deeaed0a5a99305d0d488eb952f64f,C,ScriptPromise BluetoothRemoteGATTServer::getPr...,train,119.0,True,ScriptPromise BluetoothRemoteGATTServer::getPr...,3730409132897095376
...,...,...,...,...,...,...,...,...,...,...,...
3877,CVE-2016-1621,CWE-119,Android,5a9753fca56f0eeb9f61e342b2fccffc364f9426,C,static void set_roi_map(const vpx_codec_enc_cf...,test,119.0,True,static void set_roi_map(const vpx_codec_enc_cf...,3799094739672422462
3878,CVE-2018-6063,CWE-787,Chrome,673ce95d481ea9368c4d4d43ac756ba1d6d9e608,C,MojoResult UnwrapSharedMemoryHandle(ScopedShar...,test,787.0,True,MojoResult UnwrapSharedMemoryHandle(ScopedShar...,-2818157829687982701
3879,CVE-2017-0592,CWE-119,Android,acc192347665943ca674acf117e4f74a88436922,C,"static void copyMultiCh24(short *dst, const in...",test,119.0,True,"static void copyMultiCh24(short *dst, const in...",4643769564745049032
3880,CVE-2016-1621,CWE-119,Android,04839626ed859623901ebd3a5fd483982186b59d,C,long Chapters::Atom::Parse(\n IMkvReader* p...,test,119.0,True,long Chapters::Atom::Parse( IMkvReader* pReade...,-8592900972625463354
